# KaBuM — Feature Engineering (todas as datas de coleta)

Versão adaptada do notebook original: em vez de processar **uma** pasta de
coleta por vez (`DATA_COLETA` fixo), este notebook roda a mesma extração de
features **para todas as pastas de data encontradas em `DATA_DIR`**, gerando
um `_features.csv` por categoria em cada pasta.

As funções de extração (`features_ram`, `features_cpu`, etc.) são exatamente
as mesmas do notebook original — só a orquestração mudou.

**Categorias:** CPU · GPU · Placa-mãe · SSD · Fonte · RAM

In [1]:
import re
import pathlib
from collections import defaultdict
import pandas as pd

pd.set_option("display.max_colwidth", 80)

## 1. Configuração

`DATA_DIR` deve apontar para a pasta que contém uma subpasta por data de
coleta (ex.: `00_Dados/2026-05-16/`, `00_Dados/2026-06-26/`, ...).

**Atenção:** confira se este é o mesmo `DATA_DIR` (ou `DATA_ROOT`) usado no
notebook `v2_01_modelo_preco_ram.ipynb` — no upload original os dois
apontavam para pastas de projeto *diferentes*
(`perspectiva-dado\Projetos\Projeto 1` aqui vs.
`perspectiva_dados_projeto2` no v2). Se os caminhos não baterem, os
`_features.csv` gerados aqui nunca vão ser encontrados pelo outro notebook.

In [ ]:
DATA_DIR = pathlib.Path(r"C:\Users\julia\OneDrive\Área de Trabalho\Projetos\Perspectivas de Dados\perspectiva_dados_projeto2\00_Dados")

DATAS_COLETA = sorted(p.name for p in DATA_DIR.iterdir() if p.is_dir())

categorias = ["cpu", "gpu", "placa_mae", "ssd", "fonte", "ram"]

print(f"{len(DATAS_COLETA)} pasta(s) de coleta encontradas:")
for d in DATAS_COLETA:
    print(" -", d)

7 pasta(s) de coleta encontradas:
 - 2026-05-16
 - 2026-06-26
 - 2026-06-28
 - 2026-06-30
 - 2026-07-01
 - 2026-07-02
 - 2026-07-06


## 2. Funções auxiliares

In [3]:
def extrair(nome, padrao, tipo=str, grupo=1):
    """
    Aplica um regex ao campo nome e retorna o grupo capturado.
    Retorna None se não encontrar.
    """
    if pd.isna(nome):
        return None
    match = re.search(padrao, nome, re.IGNORECASE)
    if match:
        try:
            return tipo(match.group(grupo))
        except (ValueError, IndexError):
            return None
    return None


def contem(nome, padrao):
    """Retorna True se o padrão for encontrado no nome."""
    if pd.isna(nome):
        return False
    return bool(re.search(padrao, nome, re.IGNORECASE))

## 3. RAM

In [5]:
def features_ram(df):
    d = df.copy()
    n = d["nome"]

    d["ram_geracao"] = n.apply(lambda x: extrair(x, r"(DDR[2-5])"))
    d["ram_gb"] = n.apply(lambda x: extrair(x, r"(\d+)\s*GB", int))
    d["ram_mhz"] = n.apply(lambda x: extrair(x, r"(\d{3,5})\s*MHz", int))
    d["ram_cl"] = n.apply(lambda x: extrair(x, r"CL(\d+)", int))
    d["ram_notebook"] = n.apply(lambda x: contem(x, r"Notebook|SODIMM|SO-DIMM"))

    return d

## 4. CPU

In [6]:
# Tabela de referência: socket → DDR suportado
# (usada quando o nome não informa a geração DDR)
SOCKET_DDR = {
    "AM5":     "DDR5",
    "AM4":     "DDR4",
    "AM3":     "DDR3",
    "AM2":     "DDR2",
    "FM2":     "DDR3",
    "STR5":    "DDR5",
    "SP3":     "DDR4",
    "LGA1851": "DDR5",
    "LGA1700": "DDR4/DDR5",
    "LGA1200": "DDR4",
    "LGA1151": "DDR4",
    "LGA1150": "DDR3",
    "LGA1155": "DDR3",
    "LGA2011": "DDR3/DDR4",
}

def features_cpu(df):
    d = df.copy()
    n = d["nome"]

    d["cpu_socket"] = n.apply(lambda x: extrair(x,
        r"(AM[2-5]|FM[12]|STR5|SP3"           # AMD
        r"|LGA\s*\d{3,4}"                      # LGA com prefixo
        r"|Sk(\d{3,4})"                        # Sk1151, Sk1700
        r"|(\d{3,4})[pP]\b"                    # 1151p, 1200p
        r"|L(\d{4})\b"                         # L1700
        r"|\((\d{4})\)"                        # (1851)
        r"|\b(1[0-9]\d{2}|20[0-9]{2})\s+(?:Intel|Core|i[3579]|Xeon)"  # 1700 Intel
        r")"
    ))

    def normalizar_socket(s):
        if pd.isna(s):
            return None
        s = re.sub(r"\s+", "", s).upper()
        # Remove Sk, L, p, parênteses — fica só o número
        num = re.search(r"\d{3,4}", s)
        if not num:
            return s
        n = num.group()
        # Sockets que não são LGA
        if s in ("AM2", "AM3", "AM4", "AM5", "FM1", "FM2", "STR5", "SP3"):
            return s
        # Adiciona LGA nos numéricos
        return f"LGA{n}"

    d["cpu_socket"] = d["cpu_socket"].apply(normalizar_socket)

    # ATENÇÃO: este regex captura o número inteiro que precede "W" — em SKUs
    # como "...WOF" ele acaba pegando o código do produto em vez do TDP real.
    # Mantido idêntico ao original por ora; ajustar separadamente.
    d["cpu_tdp_w"]   = n.apply(lambda x: extrair(x, r"(\d+)\s*W(?!h)", int))
    d["cpu_com_cooler"] = n.apply(lambda x: contem(x, r"Box|Wraith|Cooler|Fan"))
    d["cpu_marca"]   = n.apply(lambda x:
        "Intel" if contem(x, r"Intel|Core\s*i[3579]|Core\s*Ultra") else
        "AMD"   if contem(x, r"AMD|Ryzen|Athlon") else None
    )
    d["cpu_serie"]   = n.apply(lambda x: extrair(x, r"(Ryzen\s*[3579]|Core\s*i[3579]|Core\s*Ultra\s*\d)"))

    # DDR suportado via tabela de referência
    d["cpu_ddr_suportado"] = d["cpu_socket"].map(SOCKET_DDR)

    return d

## 5. Placa-mãe

In [7]:
def features_placa_mae(df):
    d = df.copy()
    n = d["nome"]

    d["mobo_socket"]     = n.apply(lambda x: extrair(x, r"(AM[45]|LGA\s?\d{3,4})"))
    d["mobo_socket"]     = d["mobo_socket"].str.replace(" ", "").str.upper()
    d["mobo_chipset"]    = n.apply(lambda x: extrair(x, r"\b([A-Z]\d{3,4})(?!\s*MHz)\b"))
    d["mobo_ddr"]        = n.apply(lambda x: extrair(x, r"(DDR[45])"))
    d["mobo_form_factor"]= n.apply(lambda x: extrair(x, r"\b(ATX|mATX|m-ATX|Micro-ATX|Mini-ATX|ITX|Mini-ITX)\b"))
    d["mobo_slots_m2"]   = n.apply(lambda x: extrair(x, r"(\d+)\s*x?\s*M\.2", int))
    d["mobo_max_ram_gb"] = n.apply(lambda x: extrair(x, r"(\d+)\s*GB(?=.*RAM)", int))

    return d

## 6. GPU

In [8]:
# Tabela de referência: modelo GPU → TDP aproximado (Watts)
GPU_TDP = {
    "RTX 5090": 575, "RTX 5080": 360, "RTX 5070 Ti": 300, "RTX 5070": 250,
    "RTX 5060 Ti": 180, "RTX 5060": 150,
    "RTX 4090": 450, "RTX 4080": 320, "RTX 4070 Ti": 285, "RTX 4070": 200,
    "RTX 4060 Ti": 165, "RTX 4060": 115,
    "RTX 3090": 350, "RTX 3080": 320, "RTX 3070": 220, "RTX 3060": 170,
    "RX 9070 XT": 304, "RX 9070": 220,
    "RX 7900 XTX": 355, "RX 7900 XT": 315, "RX 7800 XT": 263, "RX 7700 XT": 245,
    "RX 7600": 165, "RX 6700 XT": 230, "RX 6600": 132,
}

def extrair_modelo_gpu(nome):
    """Retorna o modelo GPU mais longo encontrado no nome."""
    if pd.isna(nome):
        return None
    for modelo in sorted(GPU_TDP.keys(), key=len, reverse=True):
        if re.search(re.escape(modelo), nome, re.IGNORECASE):
            return modelo
    # fallback: captura padrão genérico
    m = re.search(r"(RTX\s?\d{4}(?:\s?Ti)?|RX\s?\d{4}(?:\s?XT)?)", nome, re.IGNORECASE)
    return m.group(1).strip() if m else None

def features_gpu(df):
    d = df.copy()
    n = d["nome"]

    d["gpu_marca_chip"] = n.apply(lambda x:
        "NVIDIA" if contem(x, r"RTX|GTX|NVIDIA") else
        "AMD"    if contem(x, r"\bRX\b|Radeon|AMD") else
        "Intel"  if contem(x, r"Arc|Intel") else None
    )
    d["gpu_modelo"]     = n.apply(extrair_modelo_gpu)
    d["gpu_vram_gb"]    = n.apply(lambda x: extrair(x, r"(\d+)\s*GB", int))
    d["gpu_tdp_w"]      = d["gpu_modelo"].map(GPU_TDP)

    return d

## 7. SSD

In [9]:
def features_ssd(df):
    d = df.copy()
    n = d["nome"]

    d["ssd_interface"]  = n.apply(lambda x:
        "NVMe" if contem(x, r"NVMe|M\.2") else
        "SATA" if contem(x, r"SATA") else None
    )
    d["ssd_geracao_pcie"]= n.apply(lambda x: extrair(x, r"PCIe\s?(\d\.\d)|Gen\s?(\d)", grupo=1))
    d["ssd_capacidade_gb"] = n.apply(lambda x: (
        extrair(x, r"(\d+)\s*TB", float) * 1024
        if contem(x, r"\d+\s*TB") else
        extrair(x, r"(\d+)\s*GB", int)
    ))
    d["ssd_notebook"]   = n.apply(lambda x: contem(x, r"Notebook|2230|2242"))

    return d

## 8. Fonte

In [10]:
def features_fonte(df):
    d = df.copy()
    n = d["nome"]

    d["fonte_wattagem"]    = n.apply(lambda x: extrair(x, r"(\d{3,4})\s*W(?!h)", int))
    d["fonte_certificacao"]= n.apply(lambda x: extrair(x, r"(Titanium|Platinum|Gold|Silver|Bronze)"))
    d["fonte_modular"]     = n.apply(lambda x:
        "Full"  if contem(x, r"Full.?Modular|Modular\s*Full") else
        "Semi"  if contem(x, r"Semi.?Modular|Modular\s*Semi") else
        "Não"   if contem(x, r"Não.?Modular|Non.?Modular") else None
    )
    d["fonte_atx3"]        = n.apply(lambda x: contem(x, r"ATX\s*3\.0|ATX3"))

    return d

## 9. Loop principal — processa todas as datas e categorias

Para cada pasta de data em `DATAS_COLETA`, para cada categoria: lê o CSV
cru (`kabum_<cat>_<data>.csv`), aplica a função de extração correspondente,
e salva `kabum_<cat>_<data>_features.csv` na mesma pasta.

Se um arquivo não existir para uma categoria/data, ele é pulado com aviso
(igual à lógica de "colunas ausentes" do notebook v2) em vez de travar o
loop inteiro.

In [11]:
FUNCOES = {
    "cpu":       features_cpu,
    "gpu":       features_gpu,
    "placa_mae": features_placa_mae,
    "ssd":       features_ssd,
    "fonte":     features_fonte,
    "ram":       features_ram,
}

dfs_todas_datas = defaultdict(list)   # cat -> lista de DataFrames (uma por data)
resumo_geral = []

for data_coleta in DATAS_COLETA:
    pasta = DATA_DIR / data_coleta
    print(f"\n=== {data_coleta} ===")

    for cat in categorias:
        arquivo_entrada = pasta / f"kabum_{cat}_{data_coleta}.csv"
        if not arquivo_entrada.exists():
            print(f"  [pulei] {arquivo_entrada.name} não encontrado")
            continue

        df = pd.read_csv(arquivo_entrada, encoding="utf-8-sig")
        df = FUNCOES[cat](df)
        dfs_todas_datas[cat].append(df)

        arquivo_saida = pasta / f"kabum_{cat}_{data_coleta}_features.csv"
        df.to_csv(arquivo_saida, index=False, encoding="utf-8-sig")

        print(f"  {cat:10s}: {len(df):5d} produtos -> {arquivo_saida.name}")
        resumo_geral.append({"data_coleta": data_coleta, "categoria": cat, "produtos": len(df)})

print("\n✓ Feature engineering concluído para todas as datas!")
pd.DataFrame(resumo_geral)


=== 2026-05-16 ===
  cpu       :   482 produtos -> kabum_cpu_2026-05-16_features.csv
  gpu       :   629 produtos -> kabum_gpu_2026-05-16_features.csv
  placa_mae :   842 produtos -> kabum_placa_mae_2026-05-16_features.csv
  ssd       :   867 produtos -> kabum_ssd_2026-05-16_features.csv
  fonte     :   616 produtos -> kabum_fonte_2026-05-16_features.csv
  ram       :  1200 produtos -> kabum_ram_2026-05-16_features.csv

=== 2026-06-26 ===
  cpu       :   533 produtos -> kabum_cpu_2026-06-26_features.csv
  gpu       :   491 produtos -> kabum_gpu_2026-06-26_features.csv
  placa_mae :   888 produtos -> kabum_placa_mae_2026-06-26_features.csv
  ssd       :   871 produtos -> kabum_ssd_2026-06-26_features.csv
  fonte     :   594 produtos -> kabum_fonte_2026-06-26_features.csv
  ram       :  1200 produtos -> kabum_ram_2026-06-26_features.csv

=== 2026-06-28 ===
  cpu       :   531 produtos -> kabum_cpu_2026-06-28_features.csv
  gpu       :   492 produtos -> kabum_gpu_2026-06-28_features.csv


,data_coleta,categoria,produtos
0,2026-05-16,cpu,482
1,2026-05-16,gpu,629
2,2026-05-16,placa_mae,842
3,2026-05-16,ssd,867
4,2026-05-16,fonte,616
5,2026-05-16,ram,1200
6,2026-06-26,cpu,533
7,2026-06-26,gpu,491
8,2026-06-26,placa_mae,888
9,2026-06-26,ssd,871


## 10. Resumo de cobertura (todas as datas combinadas)

Mesma lógica do resumo original, mas agora agregando as linhas de **todas**
as datas processadas — dá o quadro real de cobertura que o notebook v2 vai
herdar.

In [12]:
features_por_cat = {
    "ram":       ["ram_geracao", "ram_gb", "ram_mhz", "ram_cl"],
    "cpu":       ["cpu_marca", "cpu_socket", "cpu_serie", "cpu_ddr_suportado", "cpu_tdp_w"],
    "placa_mae": ["mobo_socket", "mobo_chipset", "mobo_ddr", "mobo_form_factor"],
    "gpu":       ["gpu_marca_chip", "gpu_modelo", "gpu_vram_gb", "gpu_tdp_w"],
    "ssd":       ["ssd_interface", "ssd_capacidade_gb"],
    "fonte":     ["fonte_wattagem", "fonte_certificacao", "fonte_modular"],
}

dfs_concat = {cat: pd.concat(lista, ignore_index=True) for cat, lista in dfs_todas_datas.items()}

for cat, features in features_por_cat.items():
    if cat not in dfs_concat:
        continue
    df = dfs_concat[cat]
    total = len(df)
    print(f"\n{cat.upper()} ({total} produtos em {len(dfs_todas_datas[cat])} data(s))")
    for f in features:
        preenchidos = df[f].notna().sum()
        pct = preenchidos / total * 100
        barra = "█" * int(pct / 5) + "░" * (20 - int(pct / 5))
        print(f"  {f:<25s} {barra} {pct:5.1f}% ({preenchidos}/{total})")


RAM (8400 produtos em 7 data(s))
  ram_geracao               ██████████████████░░  91.9% (7722/8400)
  ram_gb                    ███████████████████░  99.4% (8352/8400)
  ram_mhz                   ██████████████░░░░░░  72.4% (6080/8400)
  ram_cl                    █████░░░░░░░░░░░░░░░  26.3% (2208/8400)

CPU (3682 produtos em 7 data(s))
  cpu_marca                 ███████████████████░  96.7% (3561/3682)
  cpu_socket                ████████████████░░░░  81.7% (3007/3682)
  cpu_serie                 ███████████████░░░░░  77.3% (2845/3682)
  cpu_ddr_suportado         ███████████████░░░░░  79.7% (2934/3682)
  cpu_tdp_w                 █░░░░░░░░░░░░░░░░░░░   9.6% (354/3682)

PLACA_MAE (6143 produtos em 7 data(s))
  mobo_socket               ████████████░░░░░░░░  60.4% (3710/6143)
  mobo_chipset              █████████░░░░░░░░░░░  45.7% (2810/6143)
  mobo_ddr                  ██████████████░░░░░░  74.9% (4602/6143)
  mobo_form_factor          ████████████░░░░░░░░  62.0% (3809/6143)

GPU (354